# Workshop: parallax phase retrieval on real gold (Colab T4)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](GIST_PLACEHOLDER)

Single-shot phase retrieval via the **parallax** mode of `DirectPtychography`.
Same real-gold 4D-STEM data from Hugging Face. No iterative ptycho yet — just
one direct deconvolution that gives you a usable phase image in seconds.

Two installs (same as the BF/DF/DPC workshop): `quantem.widget` (TestPyPI
prerelease) + `quantem` from the `berk-workshop` branch on `bobleesj/quantem`.

**Total runtime: 2-3 minutes on Colab T4.**

In [ ]:
!pip install -q --pre --extra-index-url https://test.pypi.org/simple/ quantem.widget huggingface_hub
!pip install -q git+https://github.com/bobleesj/quantem.git@berk-workshop

In [ ]:
import quantem as em
import quantem.widget
import torch

# cuDNN grid_sample currently fails on the parallax shift-origin step for these
# dimensions; disable to use the deterministic CUDA path (no perf impact at this
# data size). Tracked upstream.
torch.backends.cudnn.enabled = False

print("quantem        ", em.__version__)
print("quantem.widget ", quantem.widget.__version__)
print("torch          ", torch.__version__, "(cuDNN disabled)")
print("cuda available:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(no GPU)")

In [ ]:
import os, json
import numpy as np
from huggingface_hub import snapshot_download

folder = snapshot_download("bobleesj/quantem-data", repo_type="dataset",
                           allow_patterns=["4dstem/gold_512_npy_bin8/*"])
asset = os.path.join(folder, "4dstem", "gold_512_npy_bin8")
data = np.load(os.path.join(asset, "data.npy")).astype(np.float32)   # float for DirectPtychography
data = np.ascontiguousarray(data)
meta = json.load(open(os.path.join(asset, "meta.json")))

dset = em.core.datastructures.Dataset4dstem.from_array(
    data, sampling=meta["sampling"], units=meta["units"], name=meta["name"],
)
print(f"dataset: shape {dset.shape}, dtype {dset.array.dtype}")
print(f"sampling {meta['sampling']} {meta['units']}")
print(f"optics: {meta['voltage_kV']} kV, probe {meta['probe_semiangle_mrad']} mrad")

## Step 1 — Build the DirectPtychography object

`DirectPtychography.from_dataset4d` runs CoM + origin fit + auto-rotation
estimate internally. Pass the optics in SI: energy in eV, semi-angle cutoff in
radians.

In [ ]:
from quantem.diffractive_imaging import DirectPtychography

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

direct = DirectPtychography.from_dataset4d(
    dset,
    energy=meta["voltage_kV"] * 1e3,                          # 300 kV -> 300000 eV
    semiangle_cutoff=meta["probe_semiangle_mrad"] * 1e-3,     # 30 mrad -> 0.030 rad
    rotation_angle=None,                                       # auto-estimate
    device=DEVICE,
    verbose=True,
)
print(f"DirectPtychography built on {DEVICE}")

## Step 2 — Reconstruct with the parallax kernel

`deconvolution_kernel="parallax"` runs the parallax-tilt approximation (a
weighted shift-and-add over virtual BF images). Fast — single forward pass,
no iterative loop.

In [ ]:
direct.reconstruct(
    deconvolution_kernel="parallax",
    parallax_flip_phase=False,
    verbose=True,
)
print("parallax recon done")

## Step 3 — Visualize the recovered phase

The parallax phase image lives on the GPU; pull to CPU once for the widget.

In [ ]:
phase = direct.corrected_bf                       # (scan_r, scan_c) float32 on cuda
print(f"phase shape {tuple(phase.shape)}, device {phase.device}")
print(f"phase range [{phase.min().item():.3f}, {phase.max().item():.3f}]")

quantem.widget.Show2D(
    phase.detach().cpu().numpy(),
    title="Parallax phase (corrected_bf)",
    sampling=meta["sampling"][:2],
    units=meta["units"][:2],
    cmap="gray",
)

## What you just did

1. Loaded real gold 4D-STEM from Hugging Face → numpy-backed `Dataset4dstem`.
2. Built `DirectPtychography` with the gold's actual optics (300 kV, 30 mrad).
3. Ran a single-shot parallax reconstruction (no iterative loop).
4. Rendered `direct.corrected_bf` as an interactive `Show2D` widget.

Single forward pass on the T4. Companion notebook `berk_workshop_v1.ipynb`
covers browse + BF/DF + DPC on the same data.

## Try next

- `parallax_flip_phase=True` if the phase comes out inverted.
- Swap to `gold_512_npy_bin4` for a finer (4× larger) detector.
- v2 will add iterative ptychography (`PtychoLite`) for higher-resolution phase.